In [21]:
import numpy as np
import pandas as pd
import re

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

In [23]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error, median_absolute_error

## Preparacion de datos

In [24]:
df = pd.read_csv('../data/train/transformed_train.csv')

# Convert the 'Moneda' column to binary (1 for 'US$', 0 for '$')
df['Moneda'] = df['Moneda'].apply(lambda x: 1 if x == 'US$' else 0)
df.loc[df['Moneda'] == 1, 'Precio'] = df.loc[df['Moneda'] == 1, 'Precio'] * 1100

# Convert Kilometers column to numeric, removing 'km' if present
df['Kilómetros'] = df['Kilómetros'].str.replace(' km', '', regex=False).str.replace('km', '', regex=False)
df['Kilómetros'] = pd.to_numeric(df['Kilómetros'], errors='coerce')

# Handle the price conversion for US$ values (Moneda = 1)
# Assuming 1 US$ = 1100 local currency

def extraer_formato_correcto(texto):
    if isinstance(texto, str):
        match = re.search(r'\b(\d\.\d)', texto, re.IGNORECASE)
        return match.group(1) if match else np.nan
    return np.nan

# Aplicar solo sobre la columna 'Motor'
df['Motor'] = df['Motor'].apply(extraer_formato_correcto).astype(float)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14599 entries, 0 to 14598
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   idx                      14599 non-null  int64  
 1   Marca                    14599 non-null  object 
 2   Modelo                   14599 non-null  object 
 3   Versión                  14085 non-null  object 
 4   Título                   14599 non-null  object 
 5   Motor                    14432 non-null  float64
 6   Turbo                    14599 non-null  int64  
 7   cv                       7397 non-null   float64
 8   Tracción                 7020 non-null   object 
 9   Color                    14284 non-null  object 
 10  Tipo de combustible      14599 non-null  object 
 11  Puertas                  14599 non-null  float64
 12  Transmisión              14587 non-null  object 
 13  Con cámara de retroceso  3747 non-null   object 
 14  Kilómetros            

In [25]:
# Variable target
y = df["Precio"]
X = df.drop(columns=["idx", "Título", "Descripción", "Precio", "_clean"])

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Target/mean encoding (usar media de train solamente)
mean_encoded_cols = ["Marca", "Modelo", "Versión", "Color"]
for col in mean_encoded_cols:
    means = X_train.join(y_train).groupby(col)["Precio"].mean()
    X_train[col] = X_train[col].map(means)
    X_val[col] = X_val[col].map(means)  # aplicar los mismos valores (NaN si valor nuevo)

# Categóricas para one-hot
one_hot_cols = ["Tipo de combustible", "Transmisión", "Con cámara de retroceso", "Tracción", "Tipo de vendedor"]

# Normalizamos las columnas numéricas
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])  # usar la misma escala

# Ordinal encode antes de KNN imputación
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[one_hot_cols] = encoder.fit_transform(X_train[one_hot_cols])
X_val[one_hot_cols] = encoder.transform(X_val[one_hot_cols])  # usar misma codificación

# Imputación KNN solo con train
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

# Volvemos a DataFrame
X_train = pd.DataFrame(X_train_imputed, columns=X.columns, index=X_train.index)
X_val = pd.DataFrame(X_val_imputed, columns=X.columns, index=X_val.index)

# One-hot encoding (sin dummy_na)
X_train = pd.get_dummies(X_train, columns=one_hot_cols, dtype=np.float64)
X_val = pd.get_dummies(X_val, columns=one_hot_cols, dtype=np.float64)

# Alineamos columnas (puede faltar alguna en val que está en train)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (11679, 40)
Val: (2920, 40)


In [26]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11679 entries, 13091 to 7270
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Marca                        11679 non-null  float64
 1   Modelo                       11679 non-null  float64
 2   Versión                      11679 non-null  float64
 3   Motor                        11679 non-null  float64
 4   Turbo                        11679 non-null  float64
 5   cv                           11679 non-null  float64
 6   Color                        11679 non-null  float64
 7   Puertas                      11679 non-null  float64
 8   Kilómetros                   11679 non-null  float64
 9   Moneda                       11679 non-null  float64
 10  Año                          11679 non-null  float64
 11  Tipo de combustible_0.0      11679 non-null  float64
 12  Tipo de combustible_1.0      11679 non-null  float64
 13  Tipo de combustibl

In [27]:
X_train.describe()

,Marca,Modelo,Versión,Motor,Turbo,cv,Color,Puertas,Kilómetros,Moneda,...,Con cámara de retroceso_1.0,Tracción_0.0,Tracción_0.2,Tracción_0.4,Tracción_0.6,Tracción_0.8,Tracción_1.0,Tipo de vendedor_0.0,Tipo de vendedor_1.0,Tipo de vendedor_2.0
count,1.167900e+04,1.167900e+04,11679.000000,11679.000000,1.167900e+04,11679.000000,11679.000000,1.167900e+04,1.167900e+04,1.167900e+04,...,11679.000000,11679.000000,11679.000000,11679.000000,11679.000000,11679.000000,11679.000000,11679.000000,11679.000000,11679.000000
mean,-8.700027e-17,-6.144774e-17,-0.006794,0.002013,1.034269e-17,-0.078867,-0.000019,-6.083935e-19,6.570649e-17,-8.669607e-17,...,0.537974,0.444987,0.071581,0.055399,0.064218,0.073722,0.290093,0.465793,0.312527,0.221680
std,1.000043e+00,1.000043e+00,0.988912,0.998557,1.000043e+00,0.909106,0.991362,1.000043e+00,1.000043e+00,1.000043e+00,...,0.498577,0.496986,0.257805,0.228766,0.245151,0.261329,0.453825,0.498850,0.463543,0.415395
min,-1.774586e+00,-1.372223e+00,-1.595569,-1.314446,-7.297096e-01,-2.702734,-5.399380,-1.685247e-02,-6.285423e-01,-7.911525e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-5.477704e-01,-5.880098e-01,-0.471056,-0.502847,-7.297096e-01,-0.693925,0.002917,-1.304698e-02,-6.285423e-01,-7.911525e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-4.010183e-01,-2.894971e-01,-0.174023,-0.340527,-7.297096e-01,-0.362035,0.244900,-1.304698e-02,-6.269995e-02,-7.911525e-01,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,4.600633e-01,4.554959e-01,0.305413,0.308752,1.370408e+00,0.162002,0.677108,-1.304698e-02,3.354854e-01,1.263979e+00,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000
max,5.884414e+00,9.126179e+00,33.459911,7.450820,1.370408e+00,7.236504,23.710796,7.641008e+01,8.319995e+01,1.263979e+00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Base Line

In [29]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("Base Line")
mlflow.set_tracking_uri("http://localhost:5000")

In [ ]:
reports = []

## Regresor lineal

In [ ]:
# Train the model
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict on the validation set
y_pred_train_lr = lr.predict(X_train)
y_pred_val_lr = lr.predict(X_val)

In [ ]:
# Define a function to calculate all metrics and format a report
def calculate_metrics(y_train, y_pred_train, y_val, y_pred_val):
    # Calculate train metrics
    mse_train = mean_squared_error(y_train, y_pred_train)
    r2_train = r2_score(y_train, y_pred_train)
    mae_train = mean_absolute_error(y_train, y_pred_train)
    evs_train = explained_variance_score(y_train, y_pred_train)
    max_err_train = max_error(y_train, y_pred_train)
    medae_train = median_absolute_error(y_train, y_pred_train)
    
    # Calculate validation metrics
    mse_val = mean_squared_error(y_val, y_pred_val)
    r2_val = r2_score(y_val, y_pred_val)
    mae_val = mean_absolute_error(y_val, y_pred_val)
    evs_val = explained_variance_score(y_val, y_pred_val)
    max_err_val = max_error(y_val, y_pred_val)
    medae_val = median_absolute_error(y_val, y_pred_val)
    
    # Create a table format for metrics
    print("Metric       |    Train     |  Validation")
    print("-------------|--------------|---------------")
    print(f"MSE          |   {mse_train:.2e}   |   {mse_val:.2e}")
    print(f"R²           |   {r2_train:.4f}     |   {r2_val:.4f}")
    print(f"MAE          |   {mae_train:.2e}   |   {mae_val:.2e}")
    print(f"EVS          |   {evs_train:.4f}     |   {evs_val:.4f}")
    print(f"Max Error    |   {max_err_train:.2e}   |   {max_err_val:.2e}")
    print(f"Median AE    |   {medae_train:.2e}   |   {medae_val:.2e}")
    print()
    
    # Return metrics in a dictionary for reporting
    return {
        "mse_train": mse_train,
        "r2_train": r2_train,
        "mse_val": mse_val,
        "r2_val": r2_val,
        "mae_train": mae_train,
        "evs_train": evs_train,
        "max_err_train": max_err_train,
        "medae_train": medae_train,
        "mae_val": mae_val,
        "evs_val": evs_val,
        "max_err_val": max_err_val,
        "medae_val": medae_val
    }

In [ ]:
# Calculate metrics for Linear Regression model
lr_metrics = calculate_metrics(
    y_train, y_pred_train_lr, 
    y_val, y_pred_val_lr,
)

lr_metrics['name'] = "Linear Regression"
lr_metrics["model"] = lr
lr_metrics["params"] = None  # No params for Linear Regression


# Add to reports list
reports.append(lr_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   9.47e+13   |   1.37e+14
R²           |   0.8065     |   0.7365
MAE          |   5.31e+06   |   5.86e+06
EVS          |   0.8065     |   0.7366
Max Error    |   3.14e+08   |   2.88e+08
Median AE    |   3.34e+06   |   3.50e+06



## Random Forest

In [ ]:
# Definir los hiperparámetros en un diccionario
rf_params = {
    'n_estimators': 100,       # cantidad de árboles
    'max_depth': None,         # sin profundidad máxima (puede sobreajustar si no se regula)
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'random_state': 42,
    'n_jobs': -1               # usar todos los núcleos disponibles
}

In [ ]:
# Crear el modelo Random Forest
rf_reg = RandomForestRegressor(**rf_params)
rf_reg.fit(X_train, y_train)

# Predecir en el set de validación
y_pred_train_rf = rf_reg.predict(X_train)
y_pred_val_rf = rf_reg.predict(X_val)


In [ ]:
# Calcular métricas para el modelo Random Forest
rf_metrics = calculate_metrics(
    y_train, y_pred_train_rf,
    y_val, y_pred_val_rf,
)

rf_metrics["name"] = "Random Forest"
rf_metrics["model"] = rf_reg
rf_metrics["params"] = rf_params  # Agregar los parámetros del modelo al reporte

# Agregar a la lista de reportes
reports.append(rf_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.32e+13   |   6.40e+13
R²           |   0.9731     |   0.8768
MAE          |   1.25e+06   |   2.91e+06
EVS          |   0.9731     |   0.8771
Max Error    |   2.23e+08   |   2.76e+08
Median AE    |   5.66e+05   |   1.40e+06



## XGBoost

In [ ]:
# Definir los parámetros en un diccionario
xgb_params = {
    'objective': 'reg:squarederror',  # objetivo para regresión
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.1,
    'random_state': 42
}

In [ ]:
# Crear el modelo regresor con los parámetros
xgb_reg = XGBRegressor(**xgb_params)
xgb_reg.fit(X_train, y_train)

y_pred_train_xgb = xgb_reg.predict(X_train)
y_pred_val_xgb = xgb_reg.predict(X_val)


In [ ]:
# Calcular métricas para el modelo XGBoost
xgb_metrics = calculate_metrics(
    y_train, y_pred_train_xgb,
    y_val, y_pred_val_xgb
)

xgb_metrics["name"] = "XGBoost"
xgb_metrics["model"] = xgb_reg
xgb_metrics["params"] = xgb_params  # Agregar los parámetros del modelo al reporte
# Agregar a la lista de reportes
reports.append(xgb_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   3.11e+13   |   5.80e+13
R²           |   0.9365     |   0.8884
MAE          |   2.98e+06   |   3.48e+06
EVS          |   0.9365     |   0.8886
Max Error    |   3.08e+08   |   2.01e+08
Median AE    |   1.91e+06   |   1.99e+06



## Red neuronal

In [ ]:
# Definir los hiperparámetros en un diccionario
mlp_params = {
    'hidden_layer_sizes': (100, 50),   # dos capas ocultas con 100 y 50 neuronas
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.0001,                   # L2 regularization
    'learning_rate': 'adaptive',
    'max_iter': 500,
    'random_state': 42
}

In [ ]:
# Crear el modelo MLP
mlp_reg = MLPRegressor(**mlp_params)
mlp_reg.fit(X_train, y_train)

y_pred_train_mlp = mlp_reg.predict(X_train)
y_pred_val_mlp = mlp_reg.predict(X_val)

/home/elianaostro/miniconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


In [ ]:
# Calcular métricas para el modelo MLP
mlp_params_metrics = calculate_metrics(
    y_train, y_pred_train_mlp,
    y_val, y_pred_val_mlp
)

mlp_params_metrics["name"] = "MLP"
mlp_params_metrics["model"] = mlp_reg
mlp_params_metrics["params"] = mlp_params  # Agregar los parámetros del modelo al reporte

# Agregar a la lista de reportes
reports.append(mlp_params_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   8.19e+13   |   8.89e+13
R²           |   0.8382     |   0.8034
MAE          |   4.95e+06   |   5.07e+06
EVS          |   0.8382     |   0.8035
Max Error    |   3.02e+08   |   2.79e+08
Median AE    |   3.34e+06   |   3.40e+06



In [ ]:
input_example = X_train.iloc[0:1]

for report in reports:
    with mlflow.start_run(run_name=report["name"]):
        if report["params"]:
            mlflow.log_params(report["params"])
        mlflow.log_metrics({
            "mse_val": report["mse_val"],
            "r2_val": report["r2_val"],
            "mae_val": report["mae_val"],
            "evs_val": report["evs_val"],
            "max_err_val": report["max_err_val"],
            "medae_val": report["medae_val"]
        })
        mlflow.log_artifact("data/train/transformed_train.csv")
        signature = infer_signature(input_example, report["model"].predict(input_example))
        if report["name"] == "XGBoost":
            mlflow.xgboost.log_model(report["model"], "model", input_example=input_example, signature=signature)
        else:
            mlflow.sklearn.log_model(report["model"], "model", input_example=input_example, signature=signature)

2025/06/15 21:40:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/15 21:40:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: http://localhost:5000/#/experiments/795597961817543610/runs/5fa419bfbbb04d9d8c70abc58c19dba6
🧪 View experiment at: http://localhost:5000/#/experiments/795597961817543610


2025/06/15 21:40:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/795597961817543610/runs/b8d789c3499e43f3b8ef24b70b695529
🧪 View experiment at: http://localhost:5000/#/experiments/795597961817543610


/home/elianaostro/miniconda3/lib/python3.12/site-packages/xgboost/sklearn.py:1028: UserWarning: [21:40:18] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2025/06/15 21:40:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: http://localhost:5000/#/experiments/795597961817543610/runs/fb9a458e1cd9412aab08432ddc2296b2
🧪 View experiment at: http://localhost:5000/#/experiments/795597961817543610
🏃 View run MLP at: http://localhost:5000/#/experiments/795597961817543610/runs/241f690614e346d6821af576f8ff542c
🧪 View experiment at: http://localhost:5000/#/experiments/795597961817543610
